# DCL4 (A) / DRB2 (B) / DRB4 (C) / ds-RNA (D,E) Domain Contact Analysis -- per pose cluster, per backend, per interactor couple

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) -- install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

PLIP contacts from `results/rna_ds_dcl4_drb2_drb4/all_selected_summary.csv`,
produced by `workflows/postprocessing/Snakefile` (stage 3g `run_plip` + 3h
`aggregate`).

**Scope limitation, important:** `configs/rna_ds_dcl4_drb2_drb4.yaml`'s
`plip.chains` is `[['A'], ['B', 'C', 'D', 'E']]` -- receptor = **DCL4 only**,
ligand = DRB2 + DRB4 + both RNA strands combined. PLIP only reports
receptor-vs-ligand contacts, never ligand-vs-ligand ones, so this single PLIP
pass can only ever surface **DCL4-vs-X** interfaces (`reschain` is always
`A`). It structurally cannot see DRB2-DRB4, DRB2-RNA or DRB4-RNA contacts --
that would need separate PLIP passes with different `--chains` groupings
(e.g. `[['B'],['D','E']]` for DRB2-RNA). Flagging this so an *absence* of
those interfaces below isn't mistaken for a real finding -- every "one
heatmap per couple" in this notebook is one of the four DCL4-vs-X couples
this PLIP run actually captured, not every possible pairwise couple in the
complex.

**Update -- RNA-ligand pass now run:** the dedicated `plip_rna_ligands:`
pass has since completed
(`results/rna_ds_dcl4_drb2_drb4/all_selected_summary_rna_ligands.csv`,
receptor = each protein chain, ligand = RNA strands D/E). The load cell below
folds in **only its `reschain == "A"` rows**, which adds the **DCL4 x RNA(D)**
and **DCL4 x RNA(E)** couples. That file's DRB2-RNA / DRB4-RNA rows are left
out (this notebook fixes DCL4 as the receptor). **DRB2-DRB4** now has its own
third pass too (`plip_drb2_drb4:` in the config -> `all_selected_summary_drb2_drb4.csv`),
loaded in the dedicated section at the end of this notebook.

**Data provenance quirk** (see `drb2_drb4_domain_analysis.ipynb` for the full
explanation): `scripts/aggregate_summaries.py`'s "replica" column is actually
this pipeline's pose **cluster** number, and "model" is the staged **fname**.
Renamed below for clarity.

This notebook does **not** re-derive pose clusters -- those come from
`scripts/pose_cluster_anchor.py`'s rigid-anchor (DCL4) Kabsch + hierarchical
RMSD clustering (`results/rna_ds_dcl4_drb2_drb4/pose_clusters.csv`, explored
interactively in `notebooks/pose_clustering.ipynb`). Here we stratify **PLIP
domain contacts** by that same cluster label, loaded via
`selected_models.csv`.

**New in this revision:** an energy-based filter (same robust MAD approach as
`drb2_drb4_domain_analysis.ipynb`) removes numerically-unconverged
minimizations before any of the analysis below runs. Applied here too, this
finds the exact same pattern: **100% of rosettafold3's minimized structures
are numerically garbage** (energies up to 6x10^16 kJ/mol) -- rosettafold3
drops out of this notebook entirely, leaving alphafold3 and openfold3 as the
two backends analyzed below.

**Also new, and important:** a previously-unnoticed bug meant every
DCL4xDRB2/DCL4xDRB4 heatmap in every earlier version of this notebook was
silently empty. This complex needs `fix_pdb` (RNA-containing ->
pdb4amber + PDBFixer), which renumbers the whole complex continuously
across every chain instead of restarting each chain at 1 -- so PLIP's
`resnr_lig` for DRB2/DRB4 was never in the 1-based per-chain numbering the
domain tables assume (e.g. DRB2 residue 191 was reported as 1892). Fixed by
subtracting each chain's verified offset (see "Domain definitions" below)
before any domain mapping happens.

Domain boundaries (1-based inclusive -- identical residue numbering to the
sibling project since sequences were copied verbatim, see
`configs/rna_ds_dcl4_drb2_drb4.yaml`). Originally computed from UniProt's
PROSITE domain annotation. **DRB2's dsRBD2/disordered boundary is corrected
below** (87-155/156-434 -> 87-188/189-434) based on converging structural +
sequence-predictor evidence that residues 156-188 are genuinely folded, not
disordered -- see `notebooks/drb2_drb4_domain_analysis.ipynb`'s
fold-upon-binding investigation for the full evidence (near-universal
cross-backend helix formation plus a low AIUpred disorder score, both
transitioning sharply at residue 189, not 156). DCL4 and DRB4's boundaries
are unrevised.

**DCL4 (chain A)**

| DCL4 domain | Residues |
|---|---|
| helicase | 131-629 |
| DUF283 | 651-753 |
| platform | 754-931 |
| PAZ | 932-1054 |
| connector | 1055-1082 |
| RNase_IIIa | 1083-1251 |
| RNase_IIIb | 1292-1436 |
| dsRBD1 | 1462-1528 |
| linker | 1529-1620 |
| dsRBD2 | 1621-1697 |

**DRB2 (chain B)**

| DRB2 domain | Residues |
|---|---|
| dsRBD1 | 1-70 |
| linker | 71-86 |
| dsRBD2 | 87-188 (was 87-155 under the original PROSITE-based call) |
| disordered | 189-434 (was 156-434) |

**DRB4 (chain C)**

| DRB4 domain | Residues |
|---|---|
| dsRBD1 | 4-73 |
| linker | 74-81 |
| dsRBD2 | 82-150 |
| disordered | 151-291 |
| cryoEM_domain | 292-355 |

**ds-RNA (chains D/E)** -- no domains, just nucleotide position (D = 57 nt
sense strand, E = 55 nt antisense strand).


In [26]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "rna_ds_dcl4_drb2_drb4"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "DCL4"
LIGAND_CHAIN_NAMES = {"B": "DRB2", "C": "DRB4", "D": "RNA(D)", "E": "RNA(E)"}
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_PALETTE = px.colors.qualitative.Set2

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()


## Load data

In [ ]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)

# The dedicated RNA-ligand PLIP pass (configs' `plip_rna_ligands:` block -- receptor =
# each protein chain, ligand = RNA strands D/E) landed in its own summary file. Fold in
# ONLY its DCL4-as-receptor rows (reschain == RECEPTOR_CHAIN) so this notebook's "DCL4 is
# the fixed receptor" invariant still holds -- that adds the DCL4 x RNA(D) / DCL4 x RNA(E)
# couples that all_selected_summary.csv structurally cannot contain. Its DRB2-RNA / DRB4-RNA
# rows are deliberately left out here.
rna_csv = RESULTS_DIR / "all_selected_summary_rna_ligands.csv"
if rna_csv.exists():
    rna_df = pd.read_csv(rna_csv)
    rna_df = rna_df[rna_df["reschain"] == RECEPTOR_CHAIN].copy()
    print(f"{rna_csv.name}: +{len(rna_df)} DCL4-vs-RNA contact rows "
          f"({rna_df['reschain_lig'].value_counts().to_dict()})")
    df = pd.concat([df, rna_df], ignore_index=True)
else:
    print(f"{rna_csv.name} not found -- DCL4 x RNA couples stay empty "
          f"(run the plip_rna_ligands pass to populate them)")

df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv "
          f"entry (stale/partial aggregate CSV -- re-run after the full postprocessing "
          f"run finishes)")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{len(df)} contact rows total, {df['cluster'].nunique()} pose cluster(s), {n_models} model(s), "
      f"before the energy filter below")
print("contact rows per receptor-ligand chain pair:")
print(df.groupby(["reschain", "reschain_lig"]).size().rename("rows").to_frame())

## Filter out numerically-unconverged structures

Same robust (median / median-absolute-deviation) modified z-score approach
as `drb2_drb4_domain_analysis.ipynb` -- applied here, right after loading,
so every section below (per-couple heatmaps, per-cluster, per-backend) works
from the cleaned set.


In [28]:
def read_final_energy(energy_csv_path):
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)
        for line in fh:
            _, e = line.strip().split(",")
            last_energy = float(e)
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                         "final_energy": read_final_energy(energy_csv)})

energy_df = pd.DataFrame(energy_rows).dropna(subset=["final_energy"])

MOD_Z_THRESHOLD = 3.5
pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

flagged = energy_df.loc[~energy_df["energy_ok"]].merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left"
)
print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(f"Flagged {len(flagged)} / {len(energy_df)} minimized model(s) as numerically "
      f"unconverged (|modified z-score| > {MOD_Z_THRESHOLD}), by backend:")
display(flagged.groupby("backend").size().rename("n_excluded").to_frame())


Pooled final-energy median=-155,632 kJ/mol, MAD=6,250
Flagged 63 / 241 minimized model(s) as numerically unconverged (|modified z-score| > 3.5), by backend:


,n_excluded
backend,
alphafold3,4
openfold3,3
rosettafold3,56


In [29]:
good_pairs = set(zip(
    energy_df.loc[energy_df["energy_ok"], "cluster"],
    energy_df.loc[energy_df["energy_ok"], "fname"],
))
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups
df = df[df_keys.isin(good_pairs)].copy()
n_models_after = df.groupby(["cluster", "fname"]).ngroups

print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
print(df.groupby(["cluster", "backend"]).apply(lambda g: g[["fname"]].drop_duplicates().shape[0], include_groups=False)
      .rename("n_models").to_frame())


178 / 297 model(s) kept after the energy filter
                    n_models
cluster backend             
1       alphafold3        67
        openfold3         78
2       alphafold3        17
        openfold3         16


## Domain definitions

In [30]:
DCL4_DOMAINS = [
    ("helicase", 131, 629),
    ("DUF283", 651, 753),
    ("platform", 754, 931),
    ("PAZ", 932, 1054),
    ("connector", 1055, 1082),
    ("RNase_IIIa", 1083, 1251),
    ("RNase_IIIb", 1292, 1436),
    ("dsRBD1", 1462, 1528),
    ("linker", 1529, 1620),
    ("dsRBD2", 1621, 1697),
]

DRB2_DOMAINS = [
    ("dsRBD1", 1, 70),
    ("linker", 71, 86),
    ("dsRBD2", 87, 188),   # corrected from 87-155 -- see intro cell
    ("disordered", 189, 434),   # corrected from 156-434
]

DRB4_DOMAINS = [
    ("dsRBD1", 4, 73),
    ("linker", 74, 81),
    ("dsRBD2", 82, 150),
    ("disordered", 151, 291),
    ("cryoEM_domain", 292, 355),
]

def make_domain_mapper(domain_ranges):
    """domain_ranges: list of (label, start, end), inclusive on both ends.
    Returns a function mapping a pandas Series of residue numbers to domain
    labels; residues outside every range become NaN (reported separately)."""
    intervals = pd.IntervalIndex.from_tuples(
        [(start, end) for _, start, end in domain_ranges], closed="both"
    )
    labels = [label for label, _, _ in domain_ranges]

    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series(
            [labels[i] if i != -1 else pd.NA for i in idx],
            index=resnr_series.index, dtype="object",
        )
    return mapper

DCL4_LABELS = [l for l, _, _ in DCL4_DOMAINS]
DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

dcl4_mapper = make_domain_mapper(DCL4_DOMAINS)
LIGAND_DOMAINS = {"B": DRB2_DOMAINS, "C": DRB4_DOMAINS}
LIGAND_LABELS  = {"B": DRB2_LABELS, "C": DRB4_LABELS}
LIGAND_MAPPERS = {c: make_domain_mapper(d) for c, d in LIGAND_DOMAINS.items()}

df["dcl4_domain"] = dcl4_mapper(df["resnr"])

# IMPORTANT, previously-unnoticed fix: this complex needs fix_pdb (RNA-
# containing -> pdb4amber + PDBFixer), which renumbers the WHOLE complex
# continuously across every chain rather than restarting each chain at 1 --
# so PLIP's resnr_lig for DRB2/DRB4 was never in their own 1-based numbering
# our domain tables assume. Verified directly from the fixed PDBs' own
# per-chain residue ranges (identical across a 15-model random sample
# spanning every backend/seed): chain A (DCL4) 1-1701, chain B (DRB2)
# 1702-2135 (434 residues), chain C (DRB4) 2136-2490 (355 residues), chain D
# (RNA sense) 2491-2547, chain E (RNA antisense) 2548-2602. Without this
# correction, .dropna() below silently discarded essentially every DRB2/DRB4
# contact row (resnr_lig never falls inside [1,434]/[1,355]) -- every
# DCL4xDRB2/DCL4xDRB4 heatmap in every earlier version of this notebook was
# silently empty.
CHAIN_OFFSET = {"B": 1701, "C": 2135, "D": 2490, "E": 2547}
df["resnr_lig_raw"] = df["resnr_lig"]
for chain, offset in CHAIN_OFFSET.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "resnr_lig"] = df.loc[mask, "resnr_lig"] - offset

for chain, expected_len in [("B", 434), ("C", 355), ("D", 57), ("E", 55)]:
    sub = df[df["reschain_lig"] == chain]
    if sub.empty:
        continue
    bad = sub[~sub["resnr_lig"].between(1, expected_len)]
    if len(bad):
        print(f"WARNING: {len(bad)} chain-{chain} rows fall outside [1,{expected_len}] "
              f"after offset correction -- re-verify CHAIN_OFFSET['{chain}'].")
    else:
        print(f"chain {chain}: offset {CHAIN_OFFSET[chain]} verified OK "
              f"(all {len(sub)} rows land in [1,{expected_len}])")

df["ligand_domain"] = pd.NA
for chain, mapper in LIGAND_MAPPERS.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "ligand_domain"] = mapper(df.loc[mask, "resnr_lig"])

n_unmapped_r = df["dcl4_domain"].isna().sum()
print(f"Unmapped DCL4 residues (outside any domain): {n_unmapped_r} ({100*n_unmapped_r/len(df):.1f}%)")
df[["resnr", "dcl4_domain", "reschain_lig", "resnr_lig_raw", "resnr_lig", "ligand_domain"]].head(5)


chain B: offset 1701 verified OK (all 8004 rows land in [1,434])
chain C: offset 2135 verified OK (all 10248 rows land in [1,355])
Unmapped DCL4 residues (outside any domain): 3398 (18.6%)


,resnr,dcl4_domain,reschain_lig,resnr_lig_raw,resnr_lig,ligand_domain
0,4,<NA>,B,1892,191,disordered
1,28,<NA>,B,1750,49,dsRBD1
2,403,helicase,B,2134,433,disordered
3,403,helicase,B,2134,433,disordered
4,410,helicase,B,2130,429,disordered


## Interactor couples in this dataset

Every heatmap in this notebook is one **couple**: DCL4 (fixed receptor) vs.
one specific partner. Four couples are structurally possible given the
`--chains` scope limitation above (DCL4-DRB2, DCL4-DRB4, DCL4-RNA(D),
DCL4-RNA(E)) -- which of them actually have contact data is shown below,
before building any heatmaps, so an empty couple later isn't a surprise.


In [31]:
COUPLES = ["B", "C", "D", "E"]  # DRB2, DRB4, RNA(D), RNA(E)

print("Contact rows per couple (DCL4 vs. partner):")
for c in COUPLES:
    n = (df["reschain_lig"] == c).sum()
    n_models_c = df[df["reschain_lig"] == c][["cluster", "fname"]].drop_duplicates().shape[0]
    status = "POPULATED" if n else "EMPTY -- no contacts observed"
    print(f"  DCL4 x {LIGAND_CHAIN_NAMES[c]:8s}: {n:6d} contact rows across {n_models_c:3d} model(s) -- {status}")


Contact rows per couple (DCL4 vs. partner):
  DCL4 x DRB2    :   8004 contact rows across 177 model(s) -- POPULATED
  DCL4 x DRB4    :  10248 contact rows across 178 model(s) -- POPULATED
  DCL4 x RNA(D)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts observed
  DCL4 x RNA(E)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts observed


In [32]:
def domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir="domain_contacts"):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    labels = LIGAND_LABELS[ligand_chain]
    sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain", "ligand_domain"])
    if sub.empty:
        print(f"No DCL4-{ligand_name} contacts in this subset -- skipping '{title}'.")
        return None

    ct = (
        sub.groupby(["dcl4_domain", "ligand_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=DCL4_LABELS, columns=labels, fill_value=0)
    )
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=labels, y=DCL4_LABELS,
        colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate=f"DCL4 domain: %{{y}}<br>{ligand_name} domain: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model"),
    ))
    fig.update_layout(
        title=title, xaxis_title=f"{ligand_name} domain", yaxis_title="DCL4 domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(500, len(labels) * 100), height=max(500, len(DCL4_LABELS) * 40),
    )
    save_fig(fig, filename, subdir)
    return ct

def rna_contact_heatmap(data, strand, n_models_norm, title=None, filename=None, subdir="domain_contacts"):
    sub = data[(data["reschain_lig"] == strand)].dropna(subset=["dcl4_domain"])
    if sub.empty:
        print(f"No DCL4-RNA({strand}) contacts in this subset -- skipping.")
        return None

    nt_positions = sorted(sub["resnr_lig"].dropna().unique())
    ct = (
        sub.groupby(["resnr_lig", "dcl4_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=nt_positions, columns=DCL4_LABELS, fill_value=0)
    )
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DCL4_LABELS, y=[str(p) for p in nt_positions],
        colorscale="YlOrRd",
        hovertemplate=(f"DCL4 domain: %{{x}}<br>RNA nt (strand {strand}): %{{y}}"
                        "<br>%{z:.3f} contacts/model<extra></extra>"),
        colorbar=dict(title="Contacts<br>/ model"),
    ))
    fig.update_layout(
        title=title or f"DCL4 x RNA strand {strand} contact rate",
        xaxis_title="DCL4 domain", yaxis_title=f"RNA nt position (strand {strand})",
        yaxis=dict(autorange="reversed", tickfont=dict(size=8)),
        template=TEMPLATE, width=900, height=max(400, len(nt_positions) * 14),
    )
    save_fig(fig, filename or f"dcl4_rna_{strand}_heatmap.html", subdir)
    return ct

def couple_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir="domain_contacts"):
    """Dispatch to the right heatmap shape for this couple -- domain x domain
    for protein partners (B, C), nt-position x domain for RNA strands (D, E).
    One function so every section below can loop over COUPLES uniformly."""
    if ligand_chain in ("B", "C"):
        return domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir)
    return rna_contact_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir)


## One heatmap per couple (all clusters, all backends pooled)

In [33]:
n_models_total = df.groupby(["cluster", "fname"]).ngroups
ct_all = {}
for c in COUPLES:
    ct_all[c] = couple_heatmap(
        df, c, n_models_total,
        f"DCL4 x {LIGAND_CHAIN_NAMES[c]} -- all clusters, all backends (n={n_models_total} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[c].lower()}_heatmap_pooled.html",
    )


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/domain_contacts/dcl4_drb2_heatmap_pooled.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/domain_contacts/dcl4_drb4_heatmap_pooled.html


No DCL4-RNA(D) contacts in this subset -- skipping.
No DCL4-RNA(E) contacts in this subset -- skipping.


## Interaction-type breakdown (all clusters pooled)

In [34]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

print("\nInteraction type by receptor-ligand chain pair:")
display(
    df.groupby(["reschain", "reschain_lig", "interaction_type"], observed=True)
    .size().rename("count").reset_index().sort_values("count", ascending=False).head(20)
)


Interaction type counts:


,count
interaction_type,
hydrogen_bonds,9186
hydrophobic_interactions,5712
salt_bridges,3044
pi-cation_interactions,265
pi-stacking,45



Interaction type by receptor-ligand chain pair:


,reschain,reschain_lig,interaction_type,count
5,A,C,hydrogen_bonds,5001
0,A,B,hydrogen_bonds,4185
6,A,C,hydrophobic_interactions,3400
1,A,B,hydrophobic_interactions,2312
9,A,C,salt_bridges,1628
4,A,B,salt_bridges,1416
7,A,C,pi-cation_interactions,184
2,A,B,pi-cation_interactions,81
8,A,C,pi-stacking,35
3,A,B,pi-stacking,10


## Per-cluster breakdown

One section per pose cluster (`cluster` column, from `pose_cluster_anchor.py`'s
rigid-anchor Kabsch + hierarchical RMSD clustering, loaded via
`selected_models.csv`) -- each with its own couple heatmaps, so a difference
in domain-domain contacts between clusters shows up directly rather than
being averaged away in the pooled view above.


In [35]:
clusters = sorted(df["cluster"].unique())
cluster_n_models = df.groupby("cluster").apply(lambda g: g["fname"].nunique(), include_groups=False)
print(f"{len(clusters)} pose cluster(s): " +
      ", ".join(f"cluster {c} (n={cluster_n_models[c]} models)" for c in clusters))


2 pose cluster(s): cluster 1 (n=145 models), cluster 2 (n=33 models)


### Cluster 1

In [36]:
c = 1
sub_c = df[df["cluster"] == c]
n_c = cluster_n_models.get(c, 0)
print(f"cluster {c}: {n_c} model(s)")

for lig in COUPLES:
    couple_heatmap(
        sub_c, lig, n_c,
        f"DCL4 x {LIGAND_CHAIN_NAMES[lig]} -- cluster {c} (n={n_c} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_heatmap_cluster1.html",
        subdir="per_cluster",
    )


cluster 1: 145 model(s)
Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb2_heatmap_cluster1.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb4_heatmap_cluster1.html


No DCL4-RNA(D) contacts in this subset -- skipping.
No DCL4-RNA(E) contacts in this subset -- skipping.


In [37]:
print("Interaction type counts, cluster 1:")
display(df[df["cluster"] == 1]["interaction_type"].value_counts().rename("count").to_frame())


Interaction type counts, cluster 1:


,count
interaction_type,
hydrogen_bonds,7511
hydrophobic_interactions,4648
salt_bridges,2424
pi-cation_interactions,225
pi-stacking,38


### Cluster 2

In [38]:
c = 2
sub_c = df[df["cluster"] == c]
n_c = cluster_n_models.get(c, 0)
print(f"cluster {c}: {n_c} model(s)")

for lig in COUPLES:
    couple_heatmap(
        sub_c, lig, n_c,
        f"DCL4 x {LIGAND_CHAIN_NAMES[lig]} -- cluster {c} (n={n_c} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_heatmap_cluster2.html",
        subdir="per_cluster",
    )


cluster 2: 33 model(s)
Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb2_heatmap_cluster2.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb4_heatmap_cluster2.html


No DCL4-RNA(D) contacts in this subset -- skipping.
No DCL4-RNA(E) contacts in this subset -- skipping.


In [39]:
print("Interaction type counts, cluster 2:")
display(df[df["cluster"] == 2]["interaction_type"].value_counts().rename("count").to_frame())


Interaction type counts, cluster 2:


,count
interaction_type,
hydrogen_bonds,1675
hydrophobic_interactions,1064
salt_bridges,620
pi-cation_interactions,40
pi-stacking,7


## Per-backend breakdown

For each couple, a single panel figure with one heatmap per backend (shared
color scale within the panel) -- mirrors `drb2_drb4_domain_analysis.ipynb`'s
per-backend section. Only alphafold3 and openfold3 have any energy-filtered
models left (rosettafold3 dropped out entirely, see the filter section
above), so these panels are 2 columns wide rather than the up-to-6 in the
binary complex notebook.


In [40]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend").apply(lambda g: g[["fname","cluster"]].drop_duplicates().shape[0], include_groups=False)
print(f"{len(backends)} backend(s) with energy-filtered models: " +
      ", ".join(f"{b} (n={backend_n_models[b]} models)" for b in backends))


2 backend(s) with energy-filtered models: alphafold3 (n=84 models), openfold3 (n=94 models)


In [41]:
def couple_heatmap_matrix(data, ligand_chain, n_models_norm):
    """Same content as couple_heatmap() but returns (labels, y_labels, rate matrix)
    without plotting -- used to build multi-panel (per-backend / per-cluster)
    comparisons without generating (and discarding) N individual figures first."""
    if ligand_chain in ("B", "C"):
        labels = LIGAND_LABELS[ligand_chain]
        sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain", "ligand_domain"])
        if sub.empty:
            return None
        ct = (sub.groupby(["dcl4_domain", "ligand_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=DCL4_LABELS, columns=labels, fill_value=0))
        y_labels = DCL4_LABELS
    else:
        sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain"])
        if sub.empty:
            return None
        nt_positions = sorted(sub["resnr_lig"].dropna().unique())
        ct = (sub.groupby(["resnr_lig", "dcl4_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=nt_positions, columns=DCL4_LABELS, fill_value=0))
        labels, y_labels = DCL4_LABELS, [str(p) for p in nt_positions]
    return labels, y_labels, (ct / n_models_norm if n_models_norm else ct)

def backend_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_backend = {}
    for b in backends:
        res = couple_heatmap_matrix(df[df["backend"] == b], ligand_chain, backend_n_models[b])
        if res is not None:
            per_backend[b] = res
    if not per_backend:
        print(f"No DCL4-{ligand_name} contacts for any backend -- skipping panel.")
        return

    present = list(per_backend.keys())
    zmax = max(rate.values.max() for _, _, rate in per_backend.values())
    ncols = min(3, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"{b} (n={backend_n_models[b]})" for b in present],
        shared_yaxes=True,
    )
    for i, b in enumerate(present):
        labels, y_labels, rate = per_backend[b]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(
            go.Heatmap(
                z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
                zmin=0, zmax=zmax, showscale=(b == present[-1]),
                text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
                texttemplate="%{text}", textfont=dict(size=8),
                hovertemplate=f"{b}<br>DCL4: %{{y}}<br>{ligand_name}: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
                colorbar=dict(title="Mean<br>contacts/<br>model"),
            ),
            row=row, col=col,
        )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"DCL4 x {ligand_name} domain contact pairs by backend",
        template=TEMPLATE, width=max(700, 380 * ncols), height=max(460, 40 * len(y_labels if ligand_chain in ('D','E') else DCL4_LABELS)),
    )
    save_fig(fig, f"dcl4_{ligand_name.lower()}_heatmap_by_backend.html", "per_backend")

for c in COUPLES:
    backend_panel_for_couple(c)


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_backend/dcl4_drb2_heatmap_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_backend/dcl4_drb4_heatmap_by_backend.html


No DCL4-RNA(D) contacts for any backend -- skipping panel.
No DCL4-RNA(E) contacts for any backend -- skipping panel.


## Export per-cluster / per-couple domain contact tables

In [42]:
for c in clusters:
    sub_c = df[df["cluster"] == c]
    n_c = cluster_n_models[c]
    for lig in COUPLES:
        ct = couple_heatmap_matrix(sub_c, lig, n_c)
        if ct is None:
            continue
        _, _, rate = ct
        out_csv = out_path("per_cluster", f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_domain_pair_counts_cluster{c}.csv")
        rate.to_csv(out_csv)
        print(f"Saved: {out_csv}")


Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb2_domain_pair_counts_cluster1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb4_domain_pair_counts_cluster1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb2_domain_pair_counts_cluster2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4/figures/domain_analysis/per_cluster/dcl4_drb4_domain_pair_counts_cluster2.csv


---

# Manual pose clustering -- 4 quadrants of the PCA embedding

The per-cluster sections above use the pipeline's own pose clusters
(`scripts/pose_cluster_anchor.py`: rigid DCL4-anchor Kabsch + hierarchical
RMSD on the partner chains -- 2 clusters). This section is a **manual
override**: take the *same* PCA embedding those pose clusters were cut from
(`results/rna_ds_dcl4_drb2_drb4/pose_clusters.csv`'s `pc1`/`pc2`, the same
embedding `notebooks/pose_clustering.ipynb` plots) and split it into **four
quadrants** with a vertical line at `pc1 = 0` and a horizontal line at
`pc2 = 0` -- no model fitting, just the sign of each coordinate. Then re-run
the per-couple domain analysis stratified by those 4 groups.

Quadrants (edit `PC1_SPLIT` / `PC2_SPLIT` in the next cell to move the
crossing point):

| label | region |
|---|---|
| `Q1` | `pc1 >= 0`, `pc2 >= 0` |
| `Q2` | `pc1 < 0`, `pc2 >= 0` |
| `Q3` | `pc1 < 0`, `pc2 < 0` |
| `Q4` | `pc1 >= 0`, `pc2 < 0` |

The split is assigned on **all** models in `pose_clusters.csv` (every
backend); the domain heatmaps below then only see this notebook's
energy-filtered subset (AlphaFold3 + OpenFold3). The first cell assigns the
quadrants, plots the `pc1`/`pc2` scatter with the two crossing lines, and
cross-tabs the 4 quadrants against the pipeline's 2 pose clusters and the
backends -- **so the split can be eyeballed** before reading the per-quadrant
heatmaps. Figures and CSVs land in `figures/domain_analysis/per_quadrant/`.

In [ ]:
## Manual pose clustering: 4 quadrants of the pipeline's PCA embedding.
## Split pose_clusters.csv's pc1/pc2 (pose_cluster_anchor.py's anchor-Kabsch + partner-RMSF
## PCA embedding, the same one notebooks/pose_clustering.ipynb plots) with a vertical line at
## pc1=0 and a horizontal line at pc2=0 -- four quadrants, no model fitting.
COMPLEX_NAME = RESULTS_DIR.name
CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_SYMBOLS = {"alphafold3": "circle", "boltz": "square", "chai1": "diamond",
                   "openfold3": "triangle-up", "protenix": "x", "rosettafold3": "cross"}

PC1_SPLIT, PC2_SPLIT = 0.0, 0.0   # the crossing lines; edit to move the quadrant boundaries

pose = pd.read_csv(RESULTS_DIR / "pose_clusters.csv")
if "pc1" not in pose.columns:
    raise ValueError(f"{RESULTS_DIR / 'pose_clusters.csv'} has no pc1/pc2 columns -- re-run "
                     "workflows/postprocessing/Snakefile's pose_cluster rule "
                     "(needs scripts/pose_cluster_anchor.py 2026-08-24 or later).")
print(f"[{COMPLEX_NAME}] pose_clusters.csv: {len(pose)} models, "
      f"{pose['backend'].value_counts().to_dict()}")

def quadrant_label(pc1, pc2):
    right, top = pc1 >= PC1_SPLIT, pc2 >= PC2_SPLIT
    if right and top:         return "Q1 (pc1>=0, pc2>=0)"
    if not right and top:     return "Q2 (pc1<0, pc2>=0)"
    if not right and not top: return "Q3 (pc1<0, pc2<0)"
    return "Q4 (pc1>=0, pc2<0)"

pose["quadrant"] = [quadrant_label(a, b) for a, b in zip(pose["pc1"], pose["pc2"])]
QUADRANTS = sorted(pose["quadrant"].unique(), key=str)

print(f"\nquadrant split at pc1={PC1_SPLIT}, pc2={PC2_SPLIT}")
print("\nmodels per quadrant (all pose_clusters.csv rows):")
print(pose["quadrant"].value_counts().sort_index().rename("n_models").to_frame())
print("\nquadrant x pipeline pose cluster:")
print(pd.crosstab(pose["quadrant"], pose["cluster"]).to_string())
print("\nquadrant x backend:")
print(pd.crosstab(pose["quadrant"], pose["backend"]).to_string())

# --- visualise the quadrant split (same embedding as pose_clustering.ipynb's plot_pca) ---
hover_cols = ["backend", "seed", "sample_index", "cluster", "ptm", "iptm", "ranking_score"]
fig = go.Figure()
for i, cat in enumerate(QUADRANTS):
    sub = pose[pose["quadrant"] == cat]
    for backend in sorted(sub["backend"].unique()):
        bsub = sub[sub["backend"] == backend]
        fig.add_trace(go.Scatter(
            x=bsub["pc1"], y=bsub["pc2"], mode="markers",
            marker=dict(size=7, opacity=0.75,
                        color=CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)],
                        symbol=BACKEND_SYMBOLS.get(backend, "circle"),
                        line=dict(width=0.5, color="white")),
            name=f"{cat} / {backend}", customdata=bsub[hover_cols],
            hovertemplate="<br>".join(f"{c}: %{{customdata[{j}]}}" for j, c in enumerate(hover_cols))
                          + "<extra></extra>"))
fig.add_vline(x=PC1_SPLIT, line=dict(color="black", width=2, dash="dash"))
fig.add_hline(y=PC2_SPLIT, line=dict(color="black", width=2, dash="dash"))
fig.update_layout(title=f"{COMPLEX_NAME}: manual 4-quadrant split of pose PCA (pc1/pc2)",
                  xaxis_title="PC1", yaxis_title="PC2", template=TEMPLATE,
                  height=620, width=820, legend=dict(font=dict(size=9)))
save_fig(fig, f"{COMPLEX_NAME}_pose_quadrant_clusters.html", "per_quadrant")

# --- attach the quadrant label to this notebook's PLIP contact rows, joined on the raw CIF path ---
sel_key = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel_key["fname"] = sel_key["staged_cif"].apply(lambda p: Path(p).stem)
sel_key["cif_key"] = sel_key["source_cif"].str.replace(r"^results/abcfold/", "", regex=True)
sel_key = sel_key.merge(pose[["cif_path", "quadrant"]].rename(columns={"cif_path": "cif_key"}),
                        on="cif_key", how="left")
n_nolabel = sel_key["quadrant"].isna().sum()
if n_nolabel:
    print(f"\nWARNING: {n_nolabel}/{len(sel_key)} selected models had no pose_clusters.csv match")

df = df.drop(columns=[col for col in ["quadrant"] if col in df.columns])
df = df.merge(sel_key[["fname", "cluster", "quadrant"]].drop_duplicates(),
              on=["fname", "cluster"], how="left", validate="many_to_one")

quad_n_models = df.dropna(subset=["quadrant"]).groupby("quadrant")["fname"].nunique()
QUADRANTS_DF = sorted(quad_n_models.index.tolist(), key=str)
print("\nmodels per quadrant (this notebook's energy-filtered set):")
print(quad_n_models.rename("n_models").to_frame())
print(f"\ncontact rows with no quadrant label (expected 0): {df['quadrant'].isna().sum()}")

In [ ]:
## Domain contact heatmaps, one panel per quadrant
## (same layout as the "Per-backend breakdown" panel, split by quadrant instead)
def quadrant_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_quad = {}
    for q in QUADRANTS_DF:
        res = couple_heatmap_matrix(df[df["quadrant"] == q], ligand_chain, quad_n_models[q])
        if res is not None:
            per_quad[q] = res
    if not per_quad:
        print(f"No DCL4-{ligand_name} contacts for any quadrant -- skipping panel.")
        return

    present = list(per_quad.keys())
    zmax = max(rate.values.max() for _, _, rate in per_quad.values())
    ncols = min(2, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"{q} (n={quad_n_models[q]})" for q in present],
        shared_yaxes=True,
    )
    for i, q in enumerate(present):
        labels, y_labels, rate = per_quad[q]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(
            go.Heatmap(
                z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
                zmin=0, zmax=zmax, showscale=(q == present[-1]),
                text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
                texttemplate="%{text}", textfont=dict(size=8),
                hovertemplate=f"{q}<br>DCL4: %{{y}}<br>{ligand_name}: %{{x}}"
                              "<br>%{z:.3f} contacts/model<extra></extra>",
                colorbar=dict(title="Mean<br>contacts/<br>model"),
            ),
            row=row, col=col,
        )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"DCL4 x {ligand_name} domain contact pairs by pose-PCA quadrant",
        template=TEMPLATE, width=max(760, 420 * ncols), height=max(500, 320 * nrows),
    )
    save_fig(fig, f"dcl4_{ligand_name.lower()}_heatmap_by_quadrant.html", "per_quadrant")

for c in COUPLES:
    quadrant_panel_for_couple(c)

In [ ]:
## Interaction-type breakdown per quadrant
dfq = df.dropna(subset=["quadrant"])

print("Interaction type by quadrant (contact rows):")
display(pd.crosstab(dfq["quadrant"], dfq["interaction_type"]))

per_quad_itype = (
    dfq.groupby(["quadrant", "interaction_type"], observed=True).size()
       .div(quad_n_models, level="quadrant")
       .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_quad_itype, x="quadrant", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=px.colors.qualitative.Set1, template=TEMPLATE,
             category_orders={"quadrant": QUADRANTS_DF},
             title="DCL4 contacts per model, by interaction type and pose-PCA quadrant")
fig.update_layout(width=760, height=460, xaxis_title="quadrant", yaxis_title="contacts / model")
save_fig(fig, "dcl4_interaction_types_by_quadrant.html", "per_quadrant")

In [ ]:
## Export per-quadrant / per-couple domain contact-rate tables
## (copy of the "Export per-cluster tables" cell, with pose `cluster` -> `quadrant`)
for q in QUADRANTS_DF:
    sub_q = df[df["quadrant"] == q]
    n_q = quad_n_models[q]
    tag = q.split()[0].lower()   # "Q1 (pc1>=0, pc2>=0)" -> "q1"
    for lig in COUPLES:
        ct = couple_heatmap_matrix(sub_q, lig, n_q)
        if ct is None:
            continue
        _, _, rate = ct
        out_csv = out_path(
            "per_quadrant",
            f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_domain_pair_rate_{tag}.csv",
        )
        rate.to_csv(out_csv)
        print(f"Saved: {out_csv}")

---

# DRB2 x DRB4 interface -- dedicated PLIP pass

Neither PLIP pass loaded above sees the **DRB2-DRB4** interface: the main `plip:`
pass puts DRB2 (B) and DRB4 (C) *both* in the ligand group (receptor = DCL4),
and PLIP never reports ligand-vs-ligand contacts; the RNA-ligand pass is
receptor-restricted the other way. `configs/rna_ds_dcl4_drb2_drb4.yaml` now has a
third `plip_drb2_drb4:` block -- receptor = DRB2, ligand = DRB4, exactly as
`configs/drb2_drb4.yaml` / `configs/rna_ds_drb2_drb4.yaml` do for this couple in
their DCL4-free complexes -- aggregated to `all_selected_summary_drb2_drb4.csv`.

This section loads it as a standalone **DRB2 (receptor) x DRB4 (ligand)** couple:
same energy filter (`good_pairs`) and same pose-PCA quadrants as the rest of the
notebook, but its own receptor axis (DRB2 domains, not DCL4). Figures land in
`figures/domain_analysis/drb2_drb4/`.


In [ ]:
## DRB2 x DRB4 -- load the dedicated pass, energy-filter, map both sides to domains
bc_csv = RESULTS_DIR / "all_selected_summary_drb2_drb4.csv"
if not bc_csv.exists():
    raise FileNotFoundError(
        f"{bc_csv.name} not found -- run the plip_drb2_drb4 postprocessing pass:\n"
        f"  snakemake -s workflows/postprocessing/Snakefile --cores 4 --use-conda \\\n"
        f"    {bc_csv}")

df_bc = pd.read_csv(bc_csv).rename(columns={"replica": "cluster", "model": "fname"})
df_bc["cluster"] = df_bc["cluster"].astype(int)
df_bc = df_bc.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")

# same energy filter as the DCL4 couples -- reuse the good (cluster, fname) set
bc_keys = pd.MultiIndex.from_arrays([df_bc["cluster"], df_bc["fname"]])
n_bc_before = df_bc.groupby(["cluster", "fname"]).ngroups
df_bc = df_bc[bc_keys.isin(good_pairs)].copy()
n_bc_after = df_bc.groupby(["cluster", "fname"]).ngroups
print(f"{bc_csv.name}: {len(df_bc)} contact rows, {n_bc_after}/{n_bc_before} model(s) after the energy filter")
print("\nreschain x reschain_lig (expect only B x C):")
print(df_bc.groupby(["reschain", "reschain_lig"]).size().rename("rows").to_frame())

# receptor = DRB2 (chain B, continuous resnr), ligand = DRB4 (chain C) -- subtract the
# same per-chain offsets the DCL4-couple cell verified, then map to each protein's domains.
df_bc["drb2_resnr"] = df_bc["resnr"] - CHAIN_OFFSET["B"]       # 1702-2135 -> 1-434
df_bc["drb4_resnr"] = df_bc["resnr_lig"] - CHAIN_OFFSET["C"]   # 2136-2490 -> 1-355
for name, col, hi in [("DRB2", "drb2_resnr", 434), ("DRB4", "drb4_resnr", 355)]:
    bad = int((~df_bc[col].between(1, hi)).sum())
    print(f"{name}: {bad} rows outside [1,{hi}] after offset -- re-check CHAIN_OFFSET" if bad
          else f"{name}: all {len(df_bc)} rows land in [1,{hi}] (offset OK)")

df_bc["drb2_domain"] = LIGAND_MAPPERS["B"](df_bc["drb2_resnr"])
df_bc["drb4_domain"] = LIGAND_MAPPERS["C"](df_bc["drb4_resnr"])
n_unmapped = df_bc[["drb2_domain", "drb4_domain"]].isna().any(axis=1).sum()
print(f"\nrows with an unmapped domain on either side: {n_unmapped} ({100*n_unmapped/len(df_bc):.1f}%)")
df_bc[["drb2_resnr", "drb2_domain", "drb4_resnr", "drb4_domain", "interaction_type"]].head(5)

In [ ]:
## DRB2 x DRB4 -- pooled domain-pair heatmap + interaction-type breakdown
DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

def drb2_drb4_domain_matrix(data, n_models_norm):
    sub = data.dropna(subset=["drb2_domain", "drb4_domain"])
    if sub.empty:
        return None
    ct = (sub.groupby(["drb2_domain", "drb4_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=DRB4_LABELS, fill_value=0))
    return ct / n_models_norm if n_models_norm else ct

def drb2_drb4_heatmap(rate, title, filename, subdir="drb2_drb4"):
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate="DRB2 domain: %{y}<br>DRB4 domain: %{x}<br>%{z:.3f} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model")))
    fig.update_layout(title=title, xaxis_title="DRB4 domain", yaxis_title="DRB2 domain",
                      yaxis=dict(autorange="reversed"), template=TEMPLATE,
                      width=max(520, len(DRB4_LABELS) * 110), height=max(420, len(DRB2_LABELS) * 74))
    save_fig(fig, filename, subdir)

n_bc = df_bc.groupby(["cluster", "fname"]).ngroups
rate_all = drb2_drb4_domain_matrix(df_bc, n_bc)
if rate_all is None:
    print("No DRB2-DRB4 domain-mapped contacts -- nothing to plot.")
else:
    drb2_drb4_heatmap(rate_all,
        f"DRB2 x DRB4 -- all clusters, all backends pooled (n={n_bc} models)",
        "drb2_drb4_heatmap_pooled.html")
    rate_all.to_csv(out_path("drb2_drb4", "drb2_drb4_domain_pair_rate_pooled.csv"))

print("Interaction types (DRB2 x DRB4, pooled):")
display(df_bc["interaction_type"].value_counts().rename("count").to_frame())

per_backend_itype = (
    df_bc.dropna(subset=["backend"]).groupby(["backend", "interaction_type"], observed=True).size()
         .div(df_bc.groupby("backend")["fname"].nunique(), level="backend")
         .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_backend_itype, x="backend", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=px.colors.qualitative.Set1, template=TEMPLATE,
             title="DRB2 x DRB4 contacts per model, by interaction type and backend")
fig.update_layout(width=720, height=440, xaxis_title="", yaxis_title="contacts / model")
save_fig(fig, "drb2_drb4_interaction_types_by_backend.html", "drb2_drb4")

In [ ]:
## DRB2 x DRB4 -- domain-pair heatmap per pose-PCA quadrant
df_bc = df_bc.drop(columns=[c for c in ["quadrant"] if c in df_bc.columns])
df_bc = df_bc.merge(sel_key[["fname", "cluster", "quadrant"]].drop_duplicates(),
                    on=["fname", "cluster"], how="left", validate="many_to_one")
bc_quad_n = df_bc.dropna(subset=["quadrant"]).groupby("quadrant")["fname"].nunique()

mats = {}
for q in QUADRANTS_DF:
    if q not in bc_quad_n.index:
        continue
    m = drb2_drb4_domain_matrix(df_bc[df_bc["quadrant"] == q], bc_quad_n[q])
    if m is not None:
        mats[q] = m

if not mats:
    print("No DRB2-DRB4 contacts in any quadrant -- skipping panel.")
else:
    qs = list(mats)
    zmax = max(m.values.max() for m in mats.values())
    ncols = min(2, len(qs))
    nrows = -(-len(qs) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"{q} (n={bc_quad_n[q]})" for q in qs], shared_yaxes=True)
    for i, q in enumerate(qs):
        m = mats[q]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Heatmap(
            z=m.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(q == qs[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in m.values],
            texttemplate="%{text}", textfont=dict(size=8),
            hovertemplate=f"{q}<br>DRB2: %{{y}}<br>DRB4: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model")), row=row, col=col)
        m.to_csv(out_path("drb2_drb4", f"drb2_drb4_domain_pair_rate_{q.split()[0].lower()}.csv"))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title="DRB2 x DRB4 domain contact pairs by pose-PCA quadrant",
                      template=TEMPLATE, width=max(760, 420 * ncols), height=max(500, 320 * nrows))
    save_fig(fig, "drb2_drb4_heatmap_by_quadrant.html", "drb2_drb4")